# Unit 4 — Reference Solution: A City-Wide Cost of Anarchy (Tel Aviv)

> **AI-generated reference solution — read this first.**
>
> This notebook was generated by an AI agent **with the full Unit-4 context**
> (the demo notebook, the practice task, the course outline). It is a
> **reference for the *shape* of a strong answer**, not an answer key. The
> practice task is **open-ended**: a different O-D sample, distance band, or
> "bus wins" threshold can be equally defensible. Do **not** try to match these
> numbers — try to match the *quality of the loop* (DIRECT → INTERPRET →
> EXTEND) and the *honesty of the interpretation*.
>
> **It builds directly on the demo's stack and helpers** (`tdsp_path`,
> `edge_time_at`, `tdsp_weight`, the vendored `raptor`, `nearest_stop`, the
> footpath builder, `base_map`) — no new heavy dependencies. The helper cell
> reproduces them so this notebook runs **standalone** on a fresh Colab.
>
> **The honesty contract (inherited from the demo, and the whole lesson):**
> the car's time-of-day congestion is a **model**: its hourly **timing is
> grounded in real Tel Aviv data** (Ayalon Highway / TomTom Traffic Index), but
> **which roads** slow and **how deeply** are **modeled and calibrated**, not a
> measured per-segment field (the GTFS speeds were ~flat; no open per-segment
> TLV feed exists). The bus-vs-car result below is *partly* an artifact of that
> modeled congestion — every interpretation says so, and it collapses off-peak.
> "Cost of Anarchy" is a **teaching proxy**, not the game-theoretic price of
> anarchy.
>
> **Test status: RUN-ALL VERIFIED** — a fresh free-tier Colab Run-All
> (2026-07-18) executed this notebook top-to-bottom with no errors: the road
> graph built live from Overpass, the GTFS feed loaded (the FTP mirror was down,
> so the Drive-hosted fallback served it), and the interactive folium maps
> rendered. The headline numbers below are from that run — at 8am the
> car-vs-transit split is **even (bus wins 50%, 40/80)** and it collapses to
> **~0% at 11am**; the wrong-objective CoA median is **≈ 1.09**. (During dogfood
> the transit sweep was corrected to take the `min` across RAPTOR rounds — the
> true earliest arrival, not the fewest-transfers round — which is why the 8am
> bus-wins fraction is higher than an earlier cached run.)
>
> **© 2026 Ben Galon. All rights reserved.** Part of the Geo-AI course (The Arena). Provided to enrolled students for course use; not for redistribution.


## Setup

In [ ]:
# On Colab: install Unit 4 deps. Locally: assumes `uv sync --extra unit-4` was run.
import sys
if "google.colab" in sys.modules:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )
    from setup_colab import setup_unit
    setup_unit("unit-4")

## Smoke test — fail fast, before any work

In [ ]:
# --- Unit 4 smoke test (routing stack) -----------------------------------
_smoke_err = None
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import osmnx as ox
    import networkx as nx
    import geopandas as gpd
    from shapely.geometry import Point, MultiPoint
    from pyproj import Transformer
    from scipy.spatial import cKDTree
    import gtfs_kit                       # GTFS parse + clip
    import pyarrow                        # cached parquet artifacts
    import folium
    from folium import LayerControl

    _t = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)
    assert abs(_t.transform(34.78, 32.07)[0]) > 0
    _g = nx.DiGraph(); _g.add_edge(0, 1, w=1.0); _g.add_edge(1, 2, w=1.0)
    assert nx.dijkstra_path(_g, 0, 2, weight="w") == [0, 1, 2]
    assert hasattr(gtfs_kit, "read_feed")
except Exception as exc:
    _smoke_err = exc
if _smoke_err is not None:
    print("=" * 70)
    print("SMOKE TEST FAILED — environment is not ready. See KNOWN_ISSUES.md")
    print(f"  ({type(_smoke_err).__name__}: {_smoke_err})")
    print("=" * 70)
    raise _smoke_err from None

# transit backend gate (non-fatal): pyraptor is pinned <=Py3.10; vendored RAPTOR
# is the default and always runs.
USE_PYRAPTOR = False
try:
    import pyraptor  # noqa: F401
    USE_PYRAPTOR = True
except Exception:
    USE_PYRAPTOR = False
print("Smoke test passed.")
print(f"[smoke] transit backend = {'pyraptor available + vendored RAPTOR' if USE_PYRAPTOR else 'vendored RAPTOR (default)'}")

## Configuration — city, bbox, windows, data sources

In [ ]:
# --- City + bbox (same as the demo) --------------------------------------
import os, datetime as _dt
CITY = "Tel Aviv"
METRIC_CRS = "EPSG:2039"                      # Israeli TM grid (meters)
BBOX_N, BBOX_S, BBOX_E, BBOX_W = 32.135, 32.040, 34.835, 34.745
BBOX_WSEN = (BBOX_W, BBOX_S, BBOX_E, BBOX_N)  # osmnx 2.x order (W,S,E,N)
MAP_CENTER = [(BBOX_N + BBOX_S) / 2, (BBOX_E + BBOX_W) / 2]

# Departure windows we route at.
HOURS = {"8am": 8, "11am": 11, "5pm": 17}

# Hosted fallbacks (identical to the demo).
OSM_BACKUP_DRIVE_ID = "1P6SyrzDnTFld3YX2cNorpbS0JfiUhTrR"   # EPSG:2039 graphml.gz
GTFS_MIRROR_URL = "ftp://gtfs.mot.gov.il/israel-public-transportation.zip"
GTFS_RAW_DRIVE_ID = "1fb4e7XmQft-f8SG_TFhTP_OGvS8zlJkv"     # raw GTFS zip on Drive
S3_BASE_URL = ""                                            # optional pre-clipped subset
S3_GTFS_KEY = "unit4/ta-gtfs-weekday-subset.zip"

CACHE_DIR = "/content" if os.path.isdir("/content") else os.path.abspath(".")
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"City={CITY}  bbox(W,S,E,N)={BBOX_WSEN}")
print(f"windows={HOURS}  cache={CACHE_DIR}")

## Helpers — the demo's stack, reproduced unchanged

Everything below is **lifted from the demo** so this notebook is standalone. If
you ran the demo, these are the same `base_map`, `tdsp_weight`, `raptor`, etc.
The only new thing in this notebook is **scale** — running them ~100×3 times —
which is an engineering change, not an algorithm change.

In [ ]:
# --- folium scaffolding (demo) -------------------------------------------
import folium
from folium import LayerControl

def base_map(center=None, zoom=12):
    m = folium.Map(location=center or MAP_CENTER, zoom_start=zoom, tiles=None, control_scale=True)
    folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
    folium.TileLayer("CartoDB positron", name="Carto (light)").add_to(m)
    return m

def path_latlon(G, path):
    "[(lat,lon),...] for a node path (graph is projected -> back to wgs84)."
    import geopandas as gpd
    from shapely.geometry import Point
    pts = [Point(G.nodes[n]["x"], G.nodes[n]["y"]) for n in path]
    gs = gpd.GeoSeries(pts, crs=G.graph["crs"]).to_crs("EPSG:4326")
    return [(p.y, p.x) for p in gs]

def add_path(m, G, path, color, name):
    folium.PolyLine(path_latlon(G, path), color=color, weight=5, opacity=0.8,
                    tooltip=name).add_to(folium.FeatureGroup(name=name).add_to(m))
    return m

In [ ]:
# --- OSM road graph: cache -> live Overpass -> Drive backup (demo) -------
import os, gzip, shutil
import osmnx as ox
import networkx as nx

GRAPH_CACHE = os.path.join(CACHE_DIR, "ta_drive.graphml")

def _load_backup_graph():
    import gdown
    gz = os.path.join(CACHE_DIR, "ta_backup.graphml.gz")
    graphml = os.path.join(CACHE_DIR, "ta_backup.graphml")
    if not os.path.exists(graphml):
        gdown.download(id=OSM_BACKUP_DRIVE_ID, output=gz, quiet=False)
        with gzip.open(gz, "rb") as fi, open(graphml, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    return ox.load_graphml(graphml)

if os.path.exists(GRAPH_CACHE):
    G = ox.load_graphml(GRAPH_CACHE)
    print("[graph] loaded from cache.")
else:
    try:
        G = ox.graph_from_bbox(bbox=BBOX_WSEN, network_type="drive")
        G = ox.add_edge_speeds(G); G = ox.add_edge_travel_times(G)
        G = ox.project_graph(G, to_crs=METRIC_CRS)
        ox.save_graphml(G, GRAPH_CACHE)
        print("[graph] built live from Overpass and cached.")
    except Exception as e:
        print(f"[graph] live build failed ({type(e).__name__}); using backup.")
        G = _load_backup_graph()

if not all("travel_time" in d for _, _, d in G.edges(data=True)):
    G = ox.add_edge_speeds(G); G = ox.add_edge_travel_times(G)
if G.graph.get("crs") not in (METRIC_CRS, "epsg:2039", "EPSG:2039"):
    G = ox.project_graph(G, to_crs=METRIC_CRS)
print(f"[graph] {G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges  CRS={G.graph.get('crs')}")

In [ ]:
# --- GTFS fetch + clip (demo) -------------------------------------------
import urllib.request
import gtfs_kit as gk
import pandas as pd

GTFS_RAW = os.path.join(CACHE_DIR, "il_mot_gtfs.zip")

def _fetch_raw_gtfs():
    if os.path.exists(GTFS_RAW):
        return GTFS_RAW, "cache"
    try:
        print("[gtfs] fetching MoT national feed via FTP (large; one-time)...")
        urllib.request.urlretrieve(GTFS_MIRROR_URL, GTFS_RAW)
        return GTFS_RAW, "ftp"
    except Exception as e:
        print(f"[gtfs] FTP failed ({type(e).__name__}).")
    if GTFS_RAW_DRIVE_ID:
        try:
            import gdown
            print("[gtfs] falling back to Drive-hosted raw feed (gdown)...")
            gdown.download(id=GTFS_RAW_DRIVE_ID, output=GTFS_RAW, quiet=False)
            return GTFS_RAW, "drive"
        except Exception as e:
            print(f"[gtfs] Drive failed ({type(e).__name__}).")
    if S3_BASE_URL:
        urllib.request.urlretrieve(S3_BASE_URL.rstrip("/") + "/" + S3_GTFS_KEY, GTFS_RAW)
        return GTFS_RAW, "s3"
    raise RuntimeError("No GTFS available — set GTFS_RAW_DRIVE_ID or S3_BASE_URL. See KNOWN_ISSUES.md.")

def _restrict_feed(feed, *, stop_ids=None, service_ids=None):
    "Keep matching trips; cascade stops/routes/stop_times/calendar/shapes (in memory)."
    trips, st = feed.trips, feed.stop_times
    if stop_ids is not None:
        touching = st[st.stop_id.isin(stop_ids)].trip_id.unique()
        trips = trips[trips.trip_id.isin(touching)]
    if service_ids is not None:
        trips = trips[trips.service_id.isin(service_ids)]
    trips = trips.copy(); keep = set(trips.trip_id)
    feed.trips = trips
    feed.stop_times = st[st.trip_id.isin(keep)].copy()
    feed.stops = feed.stops[feed.stops.stop_id.isin(set(feed.stop_times.stop_id))].copy()
    feed.routes = feed.routes[feed.routes.route_id.isin(set(trips.route_id))].copy()
    svc = set(trips.service_id)
    if getattr(feed, "calendar", None) is not None:
        feed.calendar = feed.calendar[feed.calendar.service_id.isin(svc)].copy()
    if getattr(feed, "calendar_dates", None) is not None:
        feed.calendar_dates = feed.calendar_dates[feed.calendar_dates.service_id.isin(svc)].copy()
    if getattr(feed, "shapes", None) is not None and "shape_id" in trips.columns:
        feed.shapes = feed.shapes[feed.shapes.shape_id.isin(set(trips.shape_id.dropna()))].copy()
    return feed

def pick_weekday_service(feed):
    cal = feed.calendar.copy(); trips = feed.trips; best = None
    for day in ["monday", "tuesday", "wednesday", "thursday"]:
        sids = cal.loc[cal[day] == 1, "service_id"].tolist()
        if not sids:
            continue
        n = trips[trips.service_id.isin(sids)].shape[0]
        if best is None or n > best[2]:
            best = (sids, day, n)
    if best is None:
        sid = feed.trips.service_id.value_counts().idxmax(); return [sid], "unknown"
    return best[0], best[1]

raw, src = _fetch_raw_gtfs()
print(f"[gtfs] read_feed (source={src}; ~30s on the full feed)...")
feed = gk.read_feed(raw, dist_units="km")
_stops = feed.stops
_in_box = _stops[(_stops.stop_lat.between(BBOX_S, BBOX_N)) & (_stops.stop_lon.between(BBOX_W, BBOX_E))]
feed = _restrict_feed(feed, stop_ids=set(_in_box.stop_id))
SERVICE_IDS, WEEKDAY = pick_weekday_service(feed)
feed = _restrict_feed(feed, service_ids=set(SERVICE_IDS))
print(f"[gtfs] clipped: {len(feed.stops):,} stops, {len(feed.routes):,} routes, "
      f"{len(feed.trips):,} trips (weekday={WEEKDAY})")

In [ ]:
# --- car routing core: real-TLV-shape time-of-day w(t) + TDSP (from the demo) --
import numpy as np

def peak_multiplier(hour):
    """Arterial travel-time multiplier; hourly SHAPE grounded in REAL Tel Aviv
    congestion (deeper EVENING peak ~17:30 than morning ~08:00) - Ayalon Highway
    speeds + TomTom Traffic Index TLV. WHERE roads slow is modeled (below)."""
    h = np.asarray(hour, dtype=float)
    am = 3.6 * np.exp(-((h - 8.0) ** 2) / (2 * 1.3 ** 2))
    pm = 4.6 * np.exp(-((h - 17.5) ** 2) / (2 * 1.8 ** 2))
    return 1.0 + am + pm

_FAST = {"motorway", "trunk", "primary", "motorway_link", "trunk_link", "primary_link"}
_MID  = {"secondary", "secondary_link", "tertiary", "tertiary_link"}
def road_sensitivity(d):
    hw = d.get("highway")
    if isinstance(hw, list):
        hw = hw[0] if hw else None
    if hw in _FAST: return 1.0
    if hw in _MID:  return 0.9
    return 0.8

def edge_time_at(G, u, v, k, hour):
    "Travel time (s) for edge (u,v,k) at `hour`: real-TLV time-shape x modeled class sensitivity."
    d = G.get_edge_data(u, v)[k]
    ff = d.get("travel_time", d.get("length", 0.0) / max(d.get("speed_kph", 30), 1) * 3.6)
    excess = float(peak_multiplier(hour)) - 1.0
    return ff * (1.0 + road_sensitivity(d) * excess)

def tdsp_weight(hour):
    "A networkx weight callable freezing the clock at `hour`. Build ONCE per hour."
    def w(u, v, data):
        best = None
        for k in G.get_edge_data(u, v):
            t = edge_time_at(G, u, v, k, hour)
            if best is None or t < best:
                best = t
        return best
    return w

def tdsp_path(hour, src, dst):
    return nx.dijkstra_path(G, src, dst, weight=tdsp_weight(hour))

def total_min(path, hour):
    "Travel time (min) of a fixed node path evaluated at `hour`."
    tot = 0.0
    for u, v in zip(path[:-1], path[1:]):
        tot += min(edge_time_at(G, u, v, k, hour) for k in G.get_edge_data(u, v))
    return tot / 60.0

print("[w(t)] Car w(t) ready: real TLV time-shape (Ayalon/TomTom) x modeled class sensitivity.")

In [ ]:
# --- transit core: stop_times, footpaths, RAPTOR model + query (demo) ---
import pandas as pd
from collections import defaultdict
from scipy.spatial import cKDTree
from pyproj import Transformer

def _to_sec(s):
    try:
        h, m, sec = str(s).split(":"); return int(h) * 3600 + int(m) * 60 + int(sec)
    except Exception:
        return np.nan

ST = feed.stop_times.copy()
ST["arr_s"] = ST["arrival_time"].map(_to_sec)
ST["dep_s"] = ST["departure_time"].map(_to_sec)
ST = ST.dropna(subset=["arr_s", "dep_s"]).sort_values(["trip_id", "stop_sequence"])
ST["route_id"] = ST.trip_id.map(feed.trips.set_index("trip_id")["route_id"])
STOPS = feed.stops.set_index("stop_id")[["stop_lat", "stop_lon", "stop_name"]]

# nearest-stop lookup (projected coords + KD-tree -> vectorizable)
_tr = Transformer.from_crs("EPSG:4326", METRIC_CRS, always_xy=True)
_sxs, _sys = _tr.transform(STOPS.stop_lon.values, STOPS.stop_lat.values)
_SXY = np.column_stack([_sxs, _sys])
_STOP_IDS = np.array(STOPS.index)
_STOP_TREE = cKDTree(_SXY)
def nearest_stop_xy(x, y):
    "Nearest stop to a projected (x,y); returns (stop_id, dist_m)."
    d, i = _STOP_TREE.query([x, y])
    return _STOP_IDS[int(i)], float(d)

# footpaths within WALK_M
WALK_MPS, WALK_M = 1.2, 250
_pairs = _STOP_TREE.query_pairs(r=WALK_M)
FOOT_ADJ = defaultdict(list)
for i, j in _pairs:
    w = int(round(float(np.hypot(*(_SXY[i] - _SXY[j]))) / WALK_MPS))
    FOOT_ADJ[_STOP_IDS[i]].append((_STOP_IDS[j], w))
    FOOT_ADJ[_STOP_IDS[j]].append((_STOP_IDS[i], w))

def build_raptor_model(ST):
    routes = defaultdict(list); trip_stops = {}
    for trip_id, grp in ST.sort_values("stop_sequence").groupby("trip_id"):
        seq = [(r.stop_id, int(r.arr_s), int(r.dep_s)) for r in grp.itertuples()]
        if len(seq) < 2:
            continue
        trip_stops[trip_id] = seq
        routes[(grp.route_id.iloc[0], tuple(s for s, _, _ in seq))].append(trip_id)
    route_trips, route_stops, stop_routes = {}, {}, defaultdict(list)
    for ri, (pattern, trips) in enumerate(routes.items()):
        trips = sorted(trips, key=lambda t: trip_stops[t][0][2])
        route_trips[ri] = trips; stops = list(pattern[1]); route_stops[ri] = stops
        for pos, s in enumerate(stops):
            stop_routes[s].append((ri, pos))
    return route_trips, route_stops, stop_routes, trip_stops

print("[raptor] building route model (one-time; ~30s)...")
ROUTE_TRIPS, ROUTE_STOPS, STOP_ROUTES, TRIP_STOPS = build_raptor_model(ST)
print(f"[raptor] {len(ROUTE_TRIPS):,} patterns over {len(TRIP_STOPS):,} trips.")

def _earliest_trip(ri, pos, ready_s):
    for trip in ROUTE_TRIPS[ri]:
        if TRIP_STOPS[trip][pos][2] >= ready_s:
            return trip
    return None

def raptor(origin_stop, dest_stop, dep_s, max_rounds=5):
    "Vendored RAPTOR earliest arrival. Returns best_per_round (arrival at dest per round)."
    INF = float("inf")
    arrival = defaultdict(lambda: INF); arrival[origin_stop] = dep_s
    marked = {origin_stop}; best_per_round = []
    for k in range(max_rounds):
        queue = {}
        for s in marked:
            for ri, pos in STOP_ROUTES[s]:
                if ri not in queue or pos < queue[ri]:
                    queue[ri] = pos
        marked = set()
        for ri, start_pos in queue.items():
            stops = ROUTE_STOPS[ri]; boarded = None
            for pos in range(start_pos, len(stops)):
                s = stops[pos]
                if boarded is not None:
                    arr_here = TRIP_STOPS[boarded][pos][1]
                    if arr_here < arrival[s] and arr_here < arrival[dest_stop]:
                        arrival[s] = arr_here; marked.add(s)
                if arrival[s] < INF:
                    trip = _earliest_trip(ri, pos, arrival[s])
                    if trip is not None and (boarded is None or
                                             TRIP_STOPS[trip][pos][2] < TRIP_STOPS[boarded][pos][2]):
                        boarded = trip; start_pos = pos
        for s in list(marked):
            for nb, walk_s in FOOT_ADJ.get(s, []):
                t = arrival[s] + walk_s
                if t < arrival[nb]:
                    arrival[nb] = t; marked.add(nb)
        best_per_round.append(arrival[dest_stop] if arrival[dest_stop] < INF else None)
        if not marked:
            break
    return best_per_round

## DIRECT — sample ~100 O-D pairs

**The decision I own here:** *how* to sample, and what counts as a "real" trip.
I sample **graph nodes directly** (so every origin/destination is on the road
network — no off-graph snapping surprises) and band the **straight-line
distance to 2.5–9 km** so I am not routing trivially-short hops or
edge-of-bbox pairs. A residential-grid sample or the instructor's curated list
would be equally defensible — I write the choice down because it defines what
"my city" means in the result.

**Unreachable-by-transit convention (decided up front):** some pairs have no
bus journey in the clipped feed (RAPTOR returns all-`None`). I **keep** them and
count them as **car-wins** for framing (b), and I report the transit-reachable
denominator — silently dropping them would inflate the bus-wins fraction.

In [ ]:
import numpy as np
N_PAIRS = 80                      # modest so a full Run-All is a few minutes
DIST_LO, DIST_HI = 2500, 9000     # straight-line band (meters)

rng = np.random.default_rng(42)
nodes = np.array(list(G.nodes()))
NXY = {n: (G.nodes[n]["x"], G.nodes[n]["y"]) for n in nodes}

def od_straight_m(o, d):
    (ox_, oy), (dx, dy) = NXY[o], NXY[d]
    return float(np.hypot(ox_ - dx, oy - dy))

PAIRS = []
while len(PAIRS) < N_PAIRS:
    o, d = int(rng.choice(nodes)), int(rng.choice(nodes))
    if o != d and DIST_LO < od_straight_m(o, d) < DIST_HI:
        PAIRS.append((o, d))

# snap each origin/destination to its nearest GTFS stop ONCE (vectorized-ish).
SNAP = {}
for o, d in PAIRS:
    ox_, oy = NXY[o]; dx, dy = NXY[d]
    os_, om = nearest_stop_xy(ox_, oy)
    ds_, dm = nearest_stop_xy(dx, dy)
    SNAP[(o, d)] = (os_, om, ds_, dm)
print(f"sampled {len(PAIRS)} O-D pairs, banded {DIST_LO/1000:.1f}-{DIST_HI/1000:.1f} km straight-line")

## DIRECT — the vectorized city-scale sweep

This is the only real *engineering* step beyond the demo. For each window I
**build the TDSP weight callable once** (it is a closure — rebuilding it per
pair is wasteful) and reuse it across all pairs. Per pair I compute:

- **car (time-optimal)** = `tdsp_path(hour)` then `total_min` at that hour;
- **car (distance-optimal) re-timed at the hour** = the `weight="length"` path,
  but its travel time **evaluated at the peak** — this is the key to framing
  (a): the wrong-objective tax only shows up when you *re-time* the short route;
- **transit** = RAPTOR earliest arrival + the **access + egress walk** (so the
  comparison is door-to-door, not stop-to-stop).

In [ ]:
import pandas as pd, time

def od_sweep(hour):
    "Return a tidy frame of car/transit/CoA for every pair at departure `hour`."
    w_time = tdsp_weight(hour)          # built ONCE per hour, reused over all pairs
    rows = []
    for (o, d) in PAIRS:
        try:
            p_time = nx.dijkstra_path(G, o, d, weight=w_time)
            p_dist = nx.shortest_path(G, o, d, weight="length")
        except nx.NetworkXNoPath:
            continue
        car_time = total_min(p_time, hour)                  # time-optimal, at the hour
        dist_route_time = total_min(p_dist, hour)           # distance route, RE-TIMED at the hour
        coa_a = dist_route_time / car_time if car_time > 0 else np.nan
        os_, om, ds_, dm = SNAP[(o, d)]
        walk_min = (om + dm) / WALK_MPS / 60.0
        best = raptor(os_, ds_, hour * 3600)
        arr = min((t for t in best if t is not None), default=None)  # earliest arrival across rounds (best_per_round is monotone non-increasing)
        transit = (arr - hour * 3600) / 60.0 + walk_min if arr is not None else None
        rows.append(dict(o=o, d=d, oy=NXY[o][1], ox=NXY[o][0],
                         car_min=car_time, dist_route_min=dist_route_time, coa_a=coa_a,
                         transit_min=transit,
                         bus_wins=(transit is not None and transit < car_time),
                         transit_reachable=(transit is not None)))
    return pd.DataFrame(rows)

t0 = time.time()
sweep8 = od_sweep(8)
print(f"8am sweep: {len(sweep8)} pairs in {time.time()-t0:.1f}s")
print(sweep8[["car_min", "dist_route_min", "coa_a", "transit_min", "bus_wins"]].describe().round(2))

## INTERPRET — the 8am decision space

Two distributions and one map. **Read the numbers back in domain terms, and
state the modeled-congestion caveat.**

In [ ]:
import matplotlib.pyplot as plt, numpy as np

reach8 = sweep8[sweep8.transit_reachable]
frac_bus = sweep8.bus_wins.mean()                 # unreachable counted as car-wins
frac_tax = (sweep8.coa_a > 1.05).mean()
print(f"framing (a)  wrong-objective CoA @8am : median {sweep8.coa_a.median():.3f}, "
      f"p90 {sweep8.coa_a.quantile(0.9):.3f}; {frac_tax*100:.0f}% of pairs taxed >5%")
print(f"framing (b)  bus wins @8am            : {frac_bus*100:.0f}% of pairs "
      f"({reach8.bus_wins.sum()}/{len(sweep8)}; transit-reachable {len(reach8)}/{len(sweep8)})")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(sweep8.coa_a, bins=20, color="#d62728", alpha=0.8)
ax[0].axvline(1.0, color="k", ls="--", lw=1)
ax[0].set_title("(a) wrong-objective CoA @8am\n(distance route / time route)")
ax[0].set_xlabel("time ratio"); ax[0].set_ylabel("pairs")
ratio = (reach8.car_min / reach8.transit_min)
ax[1].hist(ratio, bins=20, color="#2ca02c", alpha=0.8)
ax[1].axvline(1.0, color="k", ls="--", lw=1)
ax[1].set_title("(b) car / transit time @8am\n(>1 = bus faster)")
ax[1].set_xlabel("car_min / transit_min"); ax[1].set_ylabel("pairs")
plt.tight_layout(); plt.show()

In [ ]:
# Map: origins coloured by bus advantage (car_min - transit_min) @8am.
import folium, branca.colormap as bcm
m = base_map(zoom=12)
adv = (reach8.car_min - reach8.transit_min)
cmap = bcm.LinearColormap(["#d62728", "#ffffbf", "#2ca02c"],
                          vmin=float(adv.min()), vmax=float(adv.max()),
                          caption="bus advantage @8am (min): green = bus faster")
fg = folium.FeatureGroup(name="origins by bus advantage @8am").add_to(m)
for _, r in reach8.iterrows():
    a = r.car_min - r.transit_min
    folium.CircleMarker(path_latlon(G, [r.o])[0], radius=6, color=None, fill=True,
                        fill_color=cmap(a), fill_opacity=0.9,
                        tooltip=f"car {r.car_min:.0f} / bus {r.transit_min:.0f} min").add_to(fg)
cmap.add_to(m); folium.LayerControl(collapsed=False).add_to(m)
m

**What this says about Tel Aviv at 8am (and the honest caveat).** With the
modeled peak on, the 8am split is **an even coin-flip** — **car and bus each
win about half** (bus wins 50% here, 40/80). Even a modeled rush-hour peak does
**not** hand the city to transit: the bus side still pays for detours,
transfers, and the access/egress walk, so its route *geometry* only pulls it
*even* with the car on cross-town pairs, not ahead. Where the **bus does win**
(~half of pairs) it clusters along **transit-dense corridors**, not by simple
north/south — `corr(origin northing, bus advantage)` is ~0, so "the center
wins" is the wrong story; "the arterial you're near wins" is closer.

For framing (a), routing by **distance** costs a **~9% median time tax** at the
peak (median CoA ≈ 1.09, p90 ≈ 1.24), and ~65% of pairs pay a tax over 5% —
smaller than you might guess, because off the arterials the distance- and
time-optimal routes often coincide.

**The caveat that the whole result rests on:** the car's congestion is a
**model** (`peak_multiplier`): its hourly timing is real Tel Aviv data, but its
spatial spread and depth are modeled, not a measured per-segment field. So "the
bus pulls even at 8am" is *partly* "the modeled peak punishes the car." The
next cell tests exactly that.

## EXTEND (a) — off-peak persistence (the designed surprise)

Re-run the sweep at **11am**, when the modeled peak is near-off. Does the
Cost of Anarchy collapse, or persist?

In [ ]:
sweep11 = od_sweep(11)
reach11 = sweep11[sweep11.transit_reachable]
print(f"bus wins  @8am: {sweep8.bus_wins.mean()*100:4.0f}%   "
      f"@11am: {sweep11.bus_wins.mean()*100:4.0f}%")
print(f"car median @8am: {sweep8.car_min.median():4.1f} min   "
      f"@11am: {sweep11.car_min.median():4.1f} min   "
      f"(ratio {sweep8.car_min.median()/sweep11.car_min.median():.1f}x = the modeled peak)")
print(f"transit median @8am: {reach8.transit_min.median():4.1f} min   "
      f"@11am: {reach11.transit_min.median():4.1f} min   (schedule-flat)")

import matplotlib.pyplot as plt, numpy as np
labels = ["8am", "11am"]
vals = [sweep8.bus_wins.mean()*100, sweep11.bus_wins.mean()*100]
fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.bar(labels, vals, color=["#2ca02c", "#9ecae1"])
for i, v in enumerate(vals):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center")
ax.set_ylabel("% of pairs where bus wins"); ax.set_title("Bus advantage collapses off-peak")
plt.tight_layout(); plt.show()

**The surprise, and the honest decomposition.** The bus-wins fraction
**collapses from ~half (50%) at 8am to ~0% at 11am**. Why? The car's median
time drops ~3.5× (30 → 9 min) while transit is **schedule-flat** (~31 min both
hours). The *entire* 8am bus advantage is the **modeled peak switching off** —
not a real off-peak congestion measurement (the feed never contained one).

So the defensible claim is **not** "the bus beats the car in Tel Aviv at rush
hour." It is: *under a modeled peak (real TLV timing, modeled spatially) that
inflates car time ~3.5×, the bus pulls even on about half of cross-town trips;
absent that peak it is not. A real finding would require a measured off-peak
congestion field — exactly the predictive `w(t)` that Unit 3's segment-speed
forecasting could supply.* The **corridors where CoA persists** at 11am (if any
in your sample) are the only part of the result that isn't the peak — map and
chase those.

## EXTEND (b) — isochrone competitiveness

Per-pair CoA answers "this trip." A **reachability isochrone** answers "this
place." Pick one origin; compare the 30-minute **car** reach (TDSP +
convex hull, as in the demo) with the **transit-reachable stops** at 8am.

In [ ]:
import networkx as nx, geopandas as gpd
from shapely.geometry import MultiPoint, Point

ISO_ORIG = PAIRS[0][0]
CUTOFF_S = 30 * 60

# car: single-source TDSP within 30 min, convex hull
lengths = nx.single_source_dijkstra_path_length(G, ISO_ORIG, cutoff=CUTOFF_S, weight=tdsp_weight(8))
car_nodes = list(lengths.keys())
hull = MultiPoint([Point(G.nodes[n]["x"], G.nodes[n]["y"]) for n in car_nodes]).convex_hull
car_poly = gpd.GeoSeries([hull], crs=G.graph["crs"]).to_crs("EPSG:4326").iloc[0]
car_km2 = gpd.GeoSeries([hull], crs=G.graph["crs"]).area.iloc[0] / 1e6

# transit: stops reachable within 30 min of the origin's nearest stop
os_, om, _, _ = SNAP[PAIRS[0]]
INF = float("inf")
from collections import defaultdict as _dd
# The single-destination raptor() returns only best_per_round for ONE dest stop,
# so it cannot expose every stop reachable within the cutoff. For an isochrone we
# need the FULL arrival map, so transit_reach() below runs the same RAPTOR loop but
# keeps (and returns) every stop reached under the time cutoff.
def transit_reach(origin_stop, dep_s, cutoff_s, max_rounds=4):
    a = _dd(lambda: INF); a[origin_stop] = dep_s; marked = {origin_stop}
    for _ in range(max_rounds):
        queue = {}
        for s in marked:
            for ri, pos in STOP_ROUTES[s]:
                if ri not in queue or pos < queue[ri]: queue[ri] = pos
        marked = set()
        for ri, sp in queue.items():
            stops = ROUTE_STOPS[ri]; bt = None
            for pos in range(sp, len(stops)):
                s = stops[pos]
                if bt is not None:
                    ah = TRIP_STOPS[bt][pos][1]
                    if ah < a[s] and ah <= dep_s + cutoff_s: a[s] = ah; marked.add(s)
                if a[s] < INF:
                    trip = _earliest_trip(ri, pos, a[s])
                    if trip is not None and (bt is None or TRIP_STOPS[trip][pos][2] < TRIP_STOPS[bt][pos][2]):
                        bt = trip; sp = pos
        for s in list(marked):
            for nb, ws in FOOT_ADJ.get(s, []):
                t = a[s] + ws
                if t < a[nb] and t <= dep_s + cutoff_s: a[nb] = t; marked.add(nb)
        if not marked: break
    return [s for s, t in a.items() if t <= dep_s + cutoff_s]

reach_stops = transit_reach(os_, 8 * 3600, CUTOFF_S)
print(f"30-min reach @8am  car hull = {car_km2:.1f} km^2 ({len(car_nodes)} nodes); "
      f"transit = {len(reach_stops)} stops")

In [ ]:
# side-by-side on one folium map
m = base_map(zoom=12)
folium.GeoJson(car_poly.__geo_interface__, name="car 30-min reach @8am",
               style_function=lambda x: {"color": "#d62728", "fillOpacity": 0.12}).add_to(m)
fg = folium.FeatureGroup(name="transit-reachable stops @8am").add_to(m)
for s in reach_stops:
    r = STOPS.loc[s]
    folium.CircleMarker([r.stop_lat, r.stop_lon], radius=2.5, color="#2ca02c",
                        fill=True, fill_opacity=0.7).add_to(fg)
folium.Marker(path_latlon(G, [ISO_ORIG])[0], icon=folium.Icon(color="blue"),
              tooltip="isochrone origin").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

**What the shapes say.** The car isochrone is a **filled blob** (continuous
road reach); the transit reach is a **dendritic set of stops along the lines**
the origin can catch. Where the green stops fan out *beyond* the red hull,
transit is *competitive in reach* (it follows a fast corridor the congested car
can't); where the red hull covers green-free neighborhoods, the car dominates.
The reach picture explains the per-pair result: the bus wins exactly the pairs
whose destination sits on a corridor the origin can reach quickly by transit.

## EXTEND (c) — wrong-class: when is "shortest path" the wrong question?

Shortest path (any mode) answers *"minimize one scalar cost between two
points."* For several real trips that framing is itself wrong, and no choice of
`w(t)` or RAPTOR rounds fixes it — you need a different model class:

| Wrong-class condition | Why shortest-path fails | What you'd need instead |
|---|---|---|
| **Destination is a polygon** (campus, mall) | "which node/stop?" is undefined; the best entrance depends on the leg | many-to-many / nearest-of-set routing; an access model |
| **Reliability matters > mean** | the median is fine but the p90 is brutal (a transfer you miss) | a travel-time **distribution**, not a point estimate; stochastic routing |
| **Parking dominates** | door-to-door cost is search-for-parking, not drive time | a parking-availability model layered on the car cost |
| **Trip is a chain** (school→work→errand) | optimizing each leg ≠ optimizing the tour | vehicle-routing / tour optimization |
| **Multi-criteria preference** (fewest transfers, least walking) | a single scalar can't express it | **McRAPTOR / Pareto** routing (the demo's sidebar) |

In each case the *output* should be a set or a distribution, not a single path —
the decision space, not the route. **Forward pointer:** the reliability case is
exactly where **U5's learned travel-time prediction** earns its keep — a GNN can
output a *predictive distribution* of segment speed that a stochastic router
consumes.

## EXTEND (d) — a learned A* heuristic *(optional, ambitious)*

The RECENT reading covers **learned graph-search heuristics** (PHIL, Pándy et
al. 2022, arXiv:2212.03978 — a GNN predicts a node-to-goal estimate used as the
A* heuristic, reportedly ~58% fewer explored nodes). Two honest caveats:
**there is no public PHIL implementation/checkpoint**, and it needs **PyTorch (a
U5 dependency)**. So this is genuinely *build-your-own / optional*.

The cell below is a **non-GNN sketch** — predict `straight-line × per-class
scale` as the heuristic — purely to make the **verification question** concrete:
**a learned heuristic that *overestimates* distance-to-goal is INADMISSIBLE and
breaks A*'s optimality guarantee.** The interesting result is not the speedup;
it is *checking the path is still optimal* (compare A* cost to Dijkstra cost).
It is guarded off by default.

In [ ]:
RUN_LEARNED_HEURISTIC = False   # set True to try the toy admissibility experiment

if RUN_LEARNED_HEURISTIC:
    import networkx as nx, numpy as np
    # A toy stand-in for a LEARNED heuristic: straight-line distance * scale, in seconds.
    # The heuristic estimates "seconds still to go". `scale` is the knob a learned model
    # would set implicitly. A heuristic that NEVER overestimates the true remaining time
    # is ADMISSIBLE -> A* stays optimal. Push `scale` up and it starts overestimating
    # (because peak congestion makes real times far larger than free-flow distance/speed),
    # becomes INADMISSIBLE, and A* may return a SUBOPTIMAL path.
    def make_h(scale):
        def h(u, target):                      # networkx calls heuristic(u, target)
            ux, uy = G.nodes[u]["x"], G.nodes[u]["y"]
            tx, ty = G.nodes[target]["x"], G.nodes[target]["y"]
            return scale * np.hypot(ux - tx, uy - ty) / (90 / 3.6)   # 90 km/h -> seconds
        return h

    # An explored-node counter so we can see the speed/optimality trade-off.
    def explored_count(o, d, h, w):
        n = {"c": 0}
        def hc(u, target):
            n["c"] += 1
            return h(u, target)
        path = nx.astar_path(G, o, d, heuristic=hc, weight=w)
        return path, n["c"]

    w_time = tdsp_weight(8)
    sample = PAIRS[:25]
    print(f"{'scale':>6} | {'avg explored':>12} | {'suboptimal pairs':>16}")
    print("-" * 42)
    for scale in [1.0, 3.0, 8.0, 20.0]:
        explored, subopt = [], 0
        for o, d in sample:
            dij = nx.dijkstra_path(G, o, d, weight=w_time)
            cost_dij = total_min(dij, 8)
            ast, nexp = explored_count(o, d, make_h(scale), w_time)
            explored.append(nexp)
            if total_min(ast, 8) > cost_dij + 1e-6:
                subopt += 1
        tag = "admissible" if scale <= 1.0 else "INADMISSIBLE"
        print(f"{scale:>6.0f} | {np.mean(explored):>12.0f} | {subopt:>6}/{len(sample)}   ({tag})")

    print("\nLesson: as `scale` rises the heuristic explores fewer nodes (faster) BUT it "
          "stops being a lower bound, and a growing fraction of pairs return a LONGER "
          "path. You MUST verify A* cost == Dijkstra cost before trusting a learned "
          "heuristic; raw speedup is not a result.")
else:
    print("Learned-heuristic experiment skipped (set RUN_LEARNED_HEURISTIC=True). "
          "Design question to answer in your log: what node features, what training "
          "labels (Dijkstra distances), and how do you GUARANTEE/CHECK admissibility?")


## Where to go next + decision-log mapping

| Loop verb | Where it lived in this notebook |
|---|---|
| **DIRECT** | sampling the O-D pairs (band + convention); the vectorized `od_sweep` (TDSP + RAPTOR + walk, named precisely) |
| **INTERPRET** | reading the 8am fractions + map back in domain terms; **stating the modeled-congestion caveat** |
| **EXTEND** | (a) the off-peak collapse + decomposition; (b) isochrone competitiveness; (c) wrong-class; (d) learned-heuristic verification |

**Honest summary to carry forward.** The headline ("the bus pulls even on
~half of cross-town trips at 8am") is *conditional on a modeled peak*. The
result that survives scrutiny is the *structure*: where the bus is competitive
(corridors, not a N/S split), how much the wrong objective costs (~9% median),
and the fact that a **measured** off-peak congestion field — the predictive
`w(t)` of Unit 3, or the learned travel-time prediction of **Unit 5** — is what
would turn this teaching proxy into a real planning result.

**Forward to Unit 5:** the abstraction choice you made here (road network with
`w(t)` vs. transit network with a schedule) is exactly the substrate decision
U5's ST-GNN inherits.